# Etapa D · Controles suficientes y costos \(J\)

**MA4703 · FitzHugh–Nagumo.** Este notebook reutiliza los módulos y resultados de A–C.

**Objetivos:** verificar las cotas de Lyapunov, comparar acoplamientos y realimentación de corrientes, romper la sincronía con corrientes balanceadas, y **evaluar** \(J\) para controles dados (no optimizar todavía).

> Reinicia el kernel si has cambiado archivos `.jl` o ejecutado `include` múltiples veces. Ejecuta las celdas en orden.

In [ ]:
using JLD2

project = dirname(Base.active_project())
include(joinpath(project,"src","FitzHughNagumoControl.jl"))
import .FitzHughNagumoControl as FHN
println("Proyecto: ",project)

## 1 · Configuración

El signo del enunciado es **\(+\lambda(v_i-\bar v)\)**: \(\lambda<0\) es atractivo. Los pesos usados después son **ejemplos** ajustables.

In [ ]:
p = FHN.FHNParameters(a=0.7,b=0.5,tau=12.5,I=1.4)
N = 12
A = FHN.complete_adjacency(N)
center = FHN.equilibrium(p)
T = 160.0
saveat = 0.25
delta = 0.01

# Fases iniciales desde el ciclo límite de la Etapa B, cuando hay caché.
bfile = joinpath(project,"data","processed","etapa_B_resultados.jld2")
if isfile(bfile)
    db = JLD2.load(bfile)
    q = db["parameters"]
    compatible = all(isapprox(getproperty(q,f),getproperty(p,f);atol=1e-12)
                     for f in (:a,:b,:tau,:I))
    if compatible
        pts = db["orbit_states"]
        x0 = zeros(2N)
        for i in 1:N
            j = 1+round(Int,(i-1)*(size(pts,2)-1)/N)
            x0[2i-1:2i] .= pts[:,j]
        end
        z_cycle = Float64.(pts[:,1])
        println("Inicialización: ciclo refinado de Etapa B")
    else
        x0 = FHN.distributed_initial_state(N,p)
        z_cycle = [-1.4,-0.2]
        println("Caché incompatible: inicialización exploratoria")
    end
else
    x0 = FHN.distributed_initial_state(N,p)
    z_cycle = [-1.4,-0.2]
    println("Sin caché: inicialización exploratoria")
end
println("E inicial = ", FHN.synchronization_error(x0))

## 2 · Acoplamiento y feedback

La garantía **global** necesita \(\lambda<-1\) y \(b>0\). Para corrientes \(u_i=-k(v_i-\bar v)\), la condición es \(k>1+\lambda_0\). Por campo medio, ambos controles son dinámicamente equivalentes cuando \(\lambda_\mathrm{efectivo}=\lambda_0-k\).

In [ ]:
lambdas = [-1.5,-0.3,-0.15,0.45]
cases = NamedTuple[]
for lam in lambdas
    net = FHN.FHNNetwork(N,p;adjacency=A,lambda=lam)
    sol = FHN.simulate_network(net;u0=x0,tspan=(0.0,T),saveat=saveat)
    metr = FHN.network_metrics(sol,net;center=center)
    push!(cases,(label="λ = $(lam)",net=net,sol=sol,metrics=metr))
end
base = FHN.FHNNetwork(N,p;adjacency=A,lambda=0.0)
fb = FHN.FHNCurrentFeedback(base;k=1.5)
solfb = FHN.simulate_feedback(fb;u0=x0,tspan=(0.0,T),saveat=saveat)
metrfb = FHN.network_metrics(solfb,base;center=center)
push!(cases,(label="Feedback k=1.5",net=base,sol=solfb,metrics=metrfb))
FHN.plot_control_comparison(cases)

## 3 · Cotas teóricas y tiempo suficiente

Para el primer caso, \(\lambda=-1.5\), se comprueba \(V(t)\le V(0)e^{-2ct}\) con \(c=\min\{-1-\lambda,b/\tau\}\). **No certifica la permanencia de oscilaciones.**

In [ ]:
cert = FHN.lyapunov_certificate(cases[1].sol,cases[1].net)
Tupper = FHN.sufficient_arrival_time(first(cert.E),delta,p;lambda=-1.5)
println("c = ",cert.c," | T suficiente = ",round(Tupper,digits=4))
println("Máxima diferencia V - cota: ",cert.max_V_violation)
FHN.plot_lyapunov_certificate(cert)

In [ ]:
cert_fb = FHN.lyapunov_certificate(solfb,fb)
println("Cota feedback: c = ",cert_fb.c,
        " | ∫Σuᵢ² dt = ",FHN.total_current_effort(solfb,fb))
FHN.plot_control_inputs(solfb,fb)

## 4 · Evaluación de \(J_s\) para controles constantes

\[
J_s=\int_0^T \left[\rho+\alpha E+\beta(1-r^2)^2+\gamma(\lambda-\lambda_{ref})^2\right]dt,\qquad E(x(T))\le\delta.
\]

Aquí \(T\) es el **primer tiempo detectado** con \(E\le\delta\), no el horizonte prefijado de simulación. Si no hay llegada, se asigna `Inf`. Sólo evaluamos controles constantes, todavía **no optimizamos**.

In [ ]:
costs = NamedTuple[]
for case in cases[1:length(lambdas)]
    j = FHN.evaluate_sync_cost(case.sol,case.net;delta=delta,
         rho=1.0,alpha=0.1,beta=0.2,gamma=0.5,
         lambda_ref=-0.15,center=center)
    push!(costs,(label=case.label,cost=j))
    println(case.label," | Estado: ",j.status,
            " | Tiempo de llegada: ",j.arrival_time,
            " | J llegada: ",j.arrival_J)
end
FHN.plot_sync_cost_traces(costs)

## 5 · Ruptura de sincronía por corrientes balanceadas

Ahora partimos de un estado **exactamente sincronizado** y añadimos \(u_i\) diferentes con \(\sum_i u_i=0\). Esto rompe la igualdad de estados; no basta para demostrar desincronización de fase sostenida.

In [ ]:
xsync = FHN.synchronized_state(N,z_cycle)
profile = FHN.balanced_current_profile(N;pattern=:split)
forced_cases = NamedTuple[]
for amp in (0.0,0.25,0.75)
    drive = FHN.balanced_current_law(profile;amplitude=amp)
    net = FHN.FHNNetwork(N,p;adjacency=A,lambda=-0.15,currents=drive)
    sol = FHN.simulate_network(net;u0=xsync,tspan=(0.0,T),saveat=saveat)
    metr = FHN.network_metrics(sol,net;center=center)
    jd = FHN.evaluate_desync_cost(sol,net;alpha_d=1.0,
                                  gamma_d=0.2,kappa=2.0,Umax=1.0,center=center)
    label = "amplitud=$(amp)"
    push!(forced_cases,(label=label,net=net,sol=sol,metrics=metr,cost=jd))
    println(label," | E(T)=",round(last(metr.E),digits=6),
            " | r(T)=",round(last(metr.r),digits=6),
            " | J_des=",round(jd.total_J,digits=4))
end
FHN.plot_control_comparison(forced_cases)

In [ ]:
FHN.plot_desync_cost_traces(forced_cases)

In [ ]:
FHN.plot_control_inputs(forced_cases[end].sol,forced_cases[end].net)

## 6 · Guardar resultados y video opcional

La Etapa D guarda matrices y metadatos en JLD2, sin serializar cierres ni el solucionador. Esto permite producir nuevos gráficos después sin repetir integraciones. El video es opcional.

In [ ]:
datadir = joinpath(project,"data","processed")
mkpath(datadir)
file = joinpath(datadir,"etapa_D_notebook_ejemplo.jld2")
lastcase = forced_cases[end]
FHN.save_control_result(file,lastcase.sol,lastcase.net,lastcase.metrics;
    label=lastcase.label,kind="notebook-desync",desync_cost=lastcase.cost)
println("Guardado: ",file)
println("Contenido: ",sort!(collect(keys(FHN.load_control_result(file)))))

In [ ]:
# Cambia a true solamente cuando quieras exportar el MP4.
crear_video = false
if crear_video
    path = joinpath(project,"docs","videos","D_02_notebook_corrientes.mp4")
    FHN.animate_network_dashboard(lastcase.sol,lastcase.net,lastcase.metrics,path;
                                  nframes=150,framerate=24)
    println("Video: ",path)
end

### Interpretación

- Un certificado de Lyapunov garantiza convergencia de **estados**, no conservación de ciclos.
- \(r\approx0\) puede corresponder a dos clústeres en oposición.
- Un \(\lambda=-0.15\) exitoso es evidencia para esas condiciones iniciales, pero no satisface la garantía global \(\lambda<-1\).
- Evaluar \(J\) para unos cuantos controles **no identifica un óptimo**. En la Etapa E buscaremos candidatos con Pontryagin y métodos numéricos directos.